![image.png](https://i.imgur.com/4fN73lZ.png)

This notebook has been inspired from [Tabular_SARSA](https://colab.research.google.com/github/probml/pyprobml/blob/master/notebooks/book2/35/supplementary/Tabular_SARSA.ipynb) by Amouzgar & Murphy and [SARSA Reinforcement Learning](https://www.geeksforgeeks.org/sarsa-reinforcement-learning/) by Alinda

### Setup

We standardize on **Gymnasium** (the maintained successor to OpenAI Gym) plus `imageio` for rendering rollouts as GIFs.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[toy-text]" imageio matplotlib

# SARSA

In this notebook, we will implement SARSA Reinforcement learning algorithm for Frozen Lake Environment.

## Frozen Lake

Frozen lake is a toy text environment involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. <br>

We can also set the lake to be slippery so that the agent does not always move in the intended direction. Here we use the non-slippery (deterministic) case for the main demo, then compare SARSA with Q-learning on a stochastic-risk task at the end.<br>

You can read more about the environment [here](https://gymnasium.farama.org/environments/toy_text/frozen_lake/).

![Frozen Lake](https://gymnasium.farama.org/_images/frozen_lake.gif)


## OpenAI Gymnasium

[OpenAI Gymnasium](https://gymnasium.farama.org/index.html) is a toolkit for developing and comparing reinforcement learning (RL) algorithms. It consists of a growing suite of environments (from simulated robots to Atari games), and a site for comparing and reproducing results. OpenAI Gymnasium provides a diverse suite of environments that range from easy to difficult and involve many different kinds of data.

Creating and Interacting with gym environments is very simple.

```
import gymnasium as gym
env = gym.make("CartPole-v1")
observation, info = env.reset(seed=42)

for _ in range(1000):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()
env.close()
```

Following are the definitions of some common terminologies used.

**Reset:** Resets the environment to an initial state and returns the initial observation. <br>
**Step:** Run one timestep of the environment's dynamics.<br>
**Observation:** The observed state of the environment.<br>
**Action:** An action provided by the agent.<br>
**Reward:** The amount of reward returned as a result of taking the action.<br>
**Terminated:** Whether a terminal state (as defined under the MDP of the task) is reached.<br>
**Truncated:** Whether a truncation condition outside the scope of the MDP is satisfied. Typically a timelimit, but could also be used to indicate agent physically going out of bounds.<br>
**Info:** This contains auxiliary diagnostic information (helpful for debugging, learning, and logging).<br>
**Action Space:** This attribute gives the format of valid actions. It is of datatype Space provided by Gym. For example, if the action space is of type Discrete and gives the value Discrete(2), this means there are two valid discrete actions: 0 & 1.<br>
**Observation:** This attribute gives the format of valid observations. It is of datatype Space provided by Gym. For example, if the observation space is of type Box and the shape of the object is (4,), this denotes a valid observation will be an array of 4 numbers.<br>

Note: Previously, `terminated` and `truncated` used to be merged under one variable `done`. <br>


We will use OpenAI Gymnasium for Frozen Lake environment.

## On-Policy vs. Off-Policy Algorithms

**On Policy:** In this, the learning agent learns the value function according to the current action derived from the policy currently being used.


**Off Policy:** In this, the learning agent learns the value function according to the action derived from another policy.

## SARSA Algorithm

SARSA algorithm is a slight variation of the Q-Learning algorithm. Q-Learning technique is an Off-Policy technique and uses the greedy approach to learn the Q-value. SARSA technique, on the other hand, is an On-Policy and uses the action performed by the current policy to learn the Q-value.

**Q-Learning:**
$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left [ r(s,a) + \gamma \max_{a'} Q(s_{t+1},a') - Q(s_t,a_t) \right ]$$

**SARSA:**
$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left [ r(s,a) + \gamma Q(s_{t+1},a_{t+1}) - Q(s_t,a_t) \right ]$$

![sarsa.png](https://i.imgur.com/6xaOkeT.png)

[Image Source](https://www.researchgate.net/publication/228410947_Adaptive_learning_by_a_target-tracking_system)

In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt

In [ ]:
# Create the environment
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

### Q-Table

Now, we need to create Q-table. A Q table helps us find the best action for each state. It gives us the Q-value for each state-action pair.<br>

To know how much rows (states) and columns (actions) we need, we need to calculate the action_size and the state_size. OpenAI Gym provides us a way to do that.

In [ ]:
state_size = env.observation_space.n
action_size = env.action_space.n

state_size, action_size

In [ ]:
# Create our Q table with state_size rows and action_size columns (64x4). We can set all values to zero for now.
qtable = np.zeros((state_size, action_size))
print(qtable)

### Exploration vs Exploitation

Notice that SARSA only learns about the states and actions it visits. What if an optimal state remains unvisited due to not being explored. The agent should sometimes pick suboptimal actions in order to visit new states and actions. <br>

A simple strategy is to use an $\epsilon$-greedy policy. According to this policy, the agent takes a random action with epsilon probability. The value of epsilon is high at the start of training and low towards the end. So, the agent explores more at the start and then exploit the learned policy more at the end.

### Hyperparameters

In [ ]:
# Here, we will specify the hyperparameters

total_episodes = 20000       # Total training episodes
learning_rate = 0.1          # Learning rate
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability
decay_rate = 0.0005           # Exponential decay rate for exploration prob

### Training

In [ ]:
def greedy_action(qtable, state):
    """Return the action with the highest Q-value in `state`."""
    return int(np.argmax(qtable[state, :]))


def epsilon_greedy(qtable, state, epsilon, env):
    """With probability `epsilon` explore (random action); otherwise act greedily."""
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    return greedy_action(qtable, state)


def train_td(env, algo, total_episodes, learning_rate, max_steps, gamma,
             max_epsilon=1.0, min_epsilon=0.01, decay_rate=0.0005):
    """Tabular temporal-difference control. `algo` selects the update target:
      - 'sarsa' (on-policy):  uses Q(s', a') for the action a' actually chosen next
      - 'q'     (off-policy): uses max_a' Q(s', a'), the best possible next action
    The two branches differ by a single line -- exactly the SARSA vs Q-learning
    distinction from the lecture. Returns the Q-table and per-episode rewards.
    """
    assert algo in ("sarsa", "q")
    qtable = np.zeros((env.observation_space.n, env.action_space.n))
    epsilon = max_epsilon
    rewards = []

    for episode in range(total_episodes):
        state, _ = env.reset()
        action = epsilon_greedy(qtable, state, epsilon, env)
        total_rewards = 0

        for step in range(max_steps):
            new_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            # SARSA needs the NEXT action under the current (behaviour) policy.
            new_action = epsilon_greedy(qtable, new_state, epsilon, env)

            if algo == "q":
                td_target = reward + gamma * np.max(qtable[new_state, :]) * (1 - done)
            else:  # sarsa
                td_target = reward + gamma * qtable[new_state, new_action] * (1 - done)
            qtable[state, action] += learning_rate * (td_target - qtable[state, action])

            total_rewards += reward
            state, action = new_state, new_action   # carry a' forward (the second 'A' in SARSA)
            if done:
                break

        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        rewards.append(total_rewards)

    return qtable, rewards


def moving_average(x, window):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")


def plot_rewards(rewards, window=500, title="Training progress"):
    """Moving-average reward. On FrozenLake (reward 1 on success, else 0) this is
    the recent success rate."""
    plt.figure(figsize=(12, 4))
    plt.plot(moving_average(rewards, window))
    plt.xlabel("Episode")
    plt.ylabel(f"Avg reward (window={window})")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
qtable, rewards = train_td(
    env, algo="sarsa",
    total_episodes=total_episodes,
    learning_rate=learning_rate,
    max_steps=max_steps,
    gamma=gamma,
    max_epsilon=max_epsilon,
    min_epsilon=min_epsilon,
    decay_rate=decay_rate,
)

print(f"Success rate over the last 1000 episodes: {np.mean(rewards[-1000:]):.3f}")
print(qtable)

plot_rewards(rewards, window=500, title="SARSA on deterministic FrozenLake")

### Visualization

In [ ]:
# Visualization helpers (Gymnasium-native, no dependency on the old `gym` package)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

In [ ]:
def record_gif(env, qtable, name, max_steps=99, fps=4):
    """Roll out the greedy policy from `qtable` and save the episode as a GIF."""
    frames = []
    state, _ = env.reset()
    for _ in range(max_steps):
        frames.append(env.render())
        state, reward, terminated, truncated, info = env.step(greedy_action(qtable, state))
        if terminated or truncated:
            frames.append(env.render())
            break
    env.close()
    path = f"video/{name}.gif"
    imageio.mimsave(path, frames, fps=fps, loop=0)
    return path

def show_gif(name):
    display(Image(filename=f"video/{name}.gif"))

In [ ]:
# Re-create the env with rendering, then watch the greedy policy
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
record_gif(env, qtable, "sarsa_frozenlake")

In [ ]:
show_gif("sarsa_frozenlake")

## On-Policy vs Off-Policy in Action: SARSA vs Q-Learning on the Cliff

The FrozenLake demo above runs SARSA on its own, but the whole point of SARSA is the
**contrast with Q-learning**. To see it we need a task where the *behaviour during
learning* matters. The classic example is **Cliff Walking** (Sutton & Barto, Example 6.6).

![Cliff Walking](https://gymnasium.farama.org/_images/cliff_walking.gif)

- A 4x12 grid. The agent starts bottom-left and must reach bottom-right.
- Every step gives reward **-1**; stepping into the cliff (the bottom row between start
  and goal) gives reward **-100** and sends the agent back to the start.
- The **optimal** path runs right along the cliff edge (shortest, return -13). But while
  the agent still explores ($\varepsilon$-greedy), walking the edge risks the occasional
  random step into the cliff.

We train both algorithms with the **same fixed** $\varepsilon = 0.1$ (no decay), so
exploration never switches off -- this is what makes the on-policy / off-policy difference
visible.

In [ ]:
# Cliff Walking is undiscounted (gamma=1) and episodic. Keep epsilon FIXED at 0.1
# (max_epsilon == min_epsilon, decay_rate = 0) so exploration stays on throughout.
cliff_kwargs = dict(
    total_episodes=2000, learning_rate=0.3, max_steps=200, gamma=1.0,
    max_epsilon=0.1, min_epsilon=0.1, decay_rate=0.0,
)

q_table_cliff, q_rewards = train_td(gym.make("CliffWalking-v1"), algo="q", **cliff_kwargs)
sarsa_table_cliff, sarsa_rewards = train_td(gym.make("CliffWalking-v1"), algo="sarsa", **cliff_kwargs)

print(f"Q-learning: average return over last 200 episodes = {np.mean(q_rewards[-200:]):.1f}")
print(f"SARSA:      average return over last 200 episodes = {np.mean(sarsa_rewards[-200:]):.1f}")

In [ ]:
# Online learning curves: the reward actually collected per episode WHILE exploring
plt.figure(figsize=(12, 4))
plt.plot(moving_average(q_rewards, 50), label="Q-learning (off-policy)")
plt.plot(moving_average(sarsa_rewards, 50), label="SARSA (on-policy)")
plt.xlabel("Episode")
plt.ylabel("Return per episode (avg over 50)")
plt.title("Cliff Walking: online performance during epsilon-greedy training")
plt.ylim(-100, 0)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Compare the GREEDY paths each algorithm converged to
def greedy_return(qtable, env, max_steps=200):
    state, _ = env.reset()
    total = 0
    for _ in range(max_steps):
        state, reward, terminated, truncated, info = env.step(greedy_action(qtable, state))
        total += reward
        if terminated or truncated:
            break
    return total

print(f"Q-learning greedy path return: {greedy_return(q_table_cliff, gym.make('CliffWalking-v1'))}")
print(f"SARSA greedy path return:      {greedy_return(sarsa_table_cliff, gym.make('CliffWalking-v1'))}")

**What to notice:**

- **Online performance** (the curve above): SARSA collects *higher* return during training.
  Because its update uses the action the $\varepsilon$-greedy policy will actually take,
  it learns that hugging the cliff edge is dangerous *while exploring* and steers one row up
  along a **safe path**.
- **Greedy path** (the returns above): Q-learning converges to the **optimal** path (return
  around -13) right along the edge, because its `max` update assumes greedy execution and
  ignores the exploration risk. SARSA's greedy path is slightly longer (around -17) but safe.
- Neither is "better" in general: Q-learning finds the optimal policy; SARSA is more robust
  when exploration (or any stochasticity) can be costly. This is exactly the
  **off-policy vs on-policy** trade-off from the lecture.

In [ ]:
# Visualize both greedy paths (Q-learning hugs the cliff edge; SARSA takes the safe detour)
record_gif(gym.make("CliffWalking-v1", render_mode="rgb_array"), q_table_cliff,
           "cliff_qlearning", max_steps=200, fps=4)
record_gif(gym.make("CliffWalking-v1", render_mode="rgb_array"), sarsa_table_cliff,
           "cliff_sarsa", max_steps=200, fps=4)
print("Q-learning greedy path:")
show_gif("cliff_qlearning")

In [ ]:
print("SARSA greedy path:")
show_gif("cliff_sarsa")